# ACUNAO Development Notebook
This is the main notebook for the development of the backend of the ACUNAO application. This notebook is created for experimentation and testing purposes. To use this notebook, follow these steps:  
1. Create a new branch on GitHub.  
2. Duplicate this notebook and make modification in the new notebook if you would like to tear the code apart, or modify in this notebook for changes agreed internally.  
3. Create a pull request.  
4. Review by another team member.  
5. Merge to main.  

WARNING: **DO NOT LEAVE THE PROCESSOR THREAD RUNNING FOREVER! PLEASE RESTART THE JUPYTER NOTEBOOK ONCE YOU ARE FINISHED WORKING ON THE NOTEBOOK.**

## PDF Parser
The following code should remain as is. Modification of the code would break the PDF parsing process.

### Method 1: Custom PDF Parser

In [1]:
# Import libraries
from typing import Optional, List, Any
import io
from pathlib import Path
import pymupdf
from PIL import Image
from pydantic import BaseModel
import cv2
import numpy as np
import pytesseract
from pytesseract import Output
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers.string import StrOutputParser
import os
from transformers import pipeline
import torch

/Users/miche/Desktop/UM/ULink AI/ACUNAO/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Element(BaseModel):
    # Custom document elements for loader
    type: str
    page_content: Any
    metadata: dict

In [3]:
def rasterize_paper(
    pdf: Path,
    outpath: Optional[Path] = None,
    dpi: int = 300,
    return_pil=False,
    pages=None,
) -> Optional[List[io.BytesIO]]:
    """
    Rasterize a PDF file to PNG images.

    Args:
        pdf (Path): The path to the PDF file.
        outpath (Optional[Path], optional): The output directory. If None, the PIL images will be returned instead. Defaults to None.
        dpi (int, optional): The output DPI. Defaults to 300.
        return_pil (bool, optional): Whether to return the PIL images instead of writing them to disk. Defaults to False.
        pages (Optional[List[int]], optional): The pages to rasterize. If None, all pages will be rasterized. Defaults to None.

    Returns:
        Optional[List[io.BytesIO]]: The PIL images if `return_pil` is True, otherwise None.
    """
    pillow_images = []
    if outpath is None:
        return_pil = True
    try:
        with pymupdf.open(pdf) as pdf_doc:
            if pages is None:
                pages = range(len(pdf_doc))
            for i in pages:
                page_bytes = pdf_doc[i].get_pixmap(dpi=dpi).tobytes("png")
                if return_pil:
                    pillow_images.append(io.BytesIO(page_bytes))
                else:
                    outpath.mkdir(parents=True, exist_ok=True)
                    with (outpath / f"{i + 1:02d}.png").open("wb") as f:
                        f.write(page_bytes)
    except Exception as e:
        print(f"Error rasterizing PDF: {e}")
        return None

    if return_pil:
        return pillow_images, pdf

In [4]:
# class PDFLoader:
#     """
#     Load PDF to a custom document element format for vector databases.

#     Args:
#         pdf_path (Path): The path to the PDF file.

#     Attributes:
#         elements: Document elements to add to vector databases.
#         pipe: Initiate table-transformer-detection.
#     """
#     def __init__(self, pdf_path: Path):
#         self.pdf_path = pdf_path
#         self.elements = []
#         self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
#         self.pipe = pipeline("object-detection", model="microsoft/table-transformer-detection", device=self.device)

#     def load(self):
#         images, filepath = rasterize_paper(self.pdf_path, return_pil=True)
#         if images is None:
#             print("Failed to rasterize PDF.")
#             return []

#         self.extract(images, filepath)
#         print("elements stored")
#         self.create_overlapping_pages(1000)
#         print("elements overlapped")
#         self.summarize_tables()
#         print("tables summarized")

#         return self.elements

#     def extract(self, images, filepath):
#         """
#         Extract tables and texts from all images.
#         """
#         for i, image in enumerate(images):
#             metadata = {"source": str(filepath), "page": i}
#             image = Image.open(image).convert("RGB")
#             results = self.pipe(image)
#             image = np.array(image)

#             boxes = []

#             for result in results:
#                 if result["label"] == "table" and result["score"] > 0.95:
#                     box = [result["box"]['xmin']-15, result["box"]['ymin']-15, result["box"]['xmax']+15, result["box"]['ymax']+15]
#                     boxes.append(box)
#                     print("boxes appended")

#                     im = image[box[1]:box[3], box[0]:box[2]]
#                     print("image cropped")

#                     # Preprocess the image for OCR
#                     im = cv2.resize(np.array(im), None, fx=1.5, fy=1.5, interpolation=cv2.INTER_CUBIC)
#                     im = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
#                     kernel = np.ones((1, 1), np.uint8)
#                     im = cv2.dilate(im, kernel, iterations=1)
#                     im = cv2.erode(im, kernel, iterations=1)
#                     im = cv2.threshold(cv2.medianBlur(im, 3), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]

#                     # Configure and conduct OCR
#                     custom_config = r'--oem 3 --psm 4'
#                     table_txt = pytesseract.image_to_string(im, config=custom_config, lang="eng")

#                     # Remove table from image
#                     cv2.rectangle(image, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (255,255,255), -1)

#                     self.elements.append(Element(type="table", page_content=table_txt, metadata=metadata))
#                     print("tables appended")

#             custom_config = r'--oem 3 --psm 1'

#             # Convert the image to grayscale
#             final_img = image[200:-200]
#             final_img = cv2.resize(final_img, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_CUBIC)

#             # Preprocess the image for OCR
#             imag = cv2.cvtColor(final_img, cv2.COLOR_BGR2GRAY)
#             kernel = np.ones((1, 1), np.uint8)
#             imag = cv2.dilate(imag, kernel, iterations=1)
#             imag = cv2.erode(imag, kernel, iterations=1)
#             imag = cv2.threshold(cv2.medianBlur(imag, 3), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
#             bboxes = self.get_paragraph_bounding_boxes(final_img)

#             texts = []
#             custom_config = r"--oem 3 --psm 1"
#             for bbox in bboxes:
#                 x, y, w, h = bbox
#                 roi = imag[y:h, x:w]

#                 fin = final_img[y:h, x:w]
#                 # Convert to grayscale and apply Otsu's threshold
#                 gray = cv2.cvtColor(fin, cv2.COLOR_BGR2GRAY)
#                 thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
#                 # Dilate with a horizontal kernel
#                 kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (20, 10))
#                 dilate = cv2.dilate(thresh, kernel, iterations=2)
#                 # Find contours
#                 cnts = cv2.findContours(dilate, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#                 cnts = cnts[0] if len(cnts) == 2 else cnts[1]

#                 contours_found = False
#                 for c in cnts:
#                     x, y, w, h = cv2.boundingRect(c)
#                     area = cv2.contourArea(c)
#                     if w/h > 2 and area > 10000:
#                         contours_found = True  # Set flag to True if contour meets criteria
#                     else:
#                         pass
                
#                 if contours_found:
#                     # Avoid appending texts from figures i.e. graph axis values
#                     text = pytesseract.image_to_string(roi, config=custom_config, lang="eng").replace("\n", " ")
#                     cleaned = text
#                     cleaned = ''.join(e for e in cleaned if e.isalnum())
#                     if cleaned.isdigit():
#                         pass
#                     else:
#                         texts.append(text)
#                 else:
#                     pass

#             if texts:
#                 texts = "\n".join(texts)
#                 self.elements.append(Element(type="text", page_content=texts, metadata=metadata))
#                 print("text appended")

#     def summarize_tables(self):
#         llm = ChatOllama(model="phi3:medium-128k", temperature=0)
#         prompt_text = """
#         You are an assistant tasked with summarizing tables. \n 
#         Give a detailed summary of the table. Do not use your pre-conceived notion and summarize exclusively with the information in the table. It is very important that you only provide the final output without any additional comments or remarks. Table: {element} 
#         """
#         prompt = ChatPromptTemplate.from_template(prompt_text)
#         summarize_chain = {"element": lambda x: x} | prompt | llm | StrOutputParser()

#         for element in self.elements:
#             if element.type == "table":
#                 summary = summarize_chain.invoke({"element": str(element.page_content)})
#                 element.page_content = summary

#     def get_paragraph_bounding_boxes(self, image):
#         # Perform OCR using Tesseract
#         custom_config = r'--oem 3 --psm 1'
#         data = pytesseract.image_to_data(image, output_type=Output.DICT, config=custom_config)
        
#         # Get bounding boxes
#         n_boxes = len(data['level'])
#         bounding_boxes = []
#         for i in range(1, n_boxes):
#             (x, y, w, h) = (data['left'][i], data['top'][i], data['width'][i], data['height'][i])
#             bounding_boxes.append((x, y, x + w, y + h))
        
#         # Merge overlapping boxes
#         def merge_boxes(boxes):
#             if not boxes:
#                 return []
            
#             boxes = sorted(boxes, key=lambda b: b[1])  # Sort by top coordinate
#             merged_boxes = [boxes[0]]
            
#             for current in boxes:
#                 last = merged_boxes[-1]
#                 if current[1] <= last[3]:  # Overlapping boxes
#                     merged_boxes[-1] = (min(last[0], current[0]), min(last[1], current[1]),
#                                         max(last[2], current[2]), max(last[3], current[3]))
#                 else:
#                     merged_boxes.append(current)
            
#             return merged_boxes
        
#         paragraphs = merge_boxes(bounding_boxes)
        
#         return paragraphs

#     def create_overlapping_pages(self, overlap_size=1000):
#         num_elements = len(self.elements)

#         for i in range(num_elements - 1):
#             if self.elements[i].metadata["source"] == "text":
#                 current_page_content = self.elements[i].page_content
#                 next_page_content = self.elements[i + 1].page_content
#                 overlap_content = next_page_content[:overlap_size]
#                 combined_content = current_page_content + ' ' + overlap_content
#                 self.elements[i].page_content = combined_content

### Method 2: Nougat as PDF Parser

In [5]:
# Import libraries
from transformers import AutoProcessor, VisionEncoderDecoderModel, StoppingCriteria, StoppingCriteriaList
import torch
from collections import defaultdict
from PIL import Image
import re

In [7]:
class RunningVarTorch: 
    def __init__(self, L=15, norm=False):
        self.values = None
        self.L = L
        self.norm = norm

    def push(self, x: torch.Tensor):
        assert x.dim() == 1
        if self.values is None: 
            self.values = x[:, None]
        elif self.values.shape[1] < self.L:
            self.values = torch.cat((self.values, x[:, None]), 1)
        else:
            self.values = torch.cat((self.values[:, 1:], x[:, None]), 1)

    def variance(self):
        if self.values is None:
            return
        if self.norm:
            return torch.var(self.values, 1) / self.values.shape[1]
        else:
            return torch.var(self.values, 1)

In [8]:
class StoppingCriteriaScores(StoppingCriteria):
    def __init__(self, threshold: float = 0.015, window_size: int = 200):
        super().__init__()
        self.threshold = threshold
        self.vars = RunningVarTorch(norm=True)
        self.varvars = RunningVarTorch(L=window_size)
        self.stop_inds = defaultdict(int)
        self.stopped = defaultdict(bool)
        self.size = 0
        self.window_size = window_size

    @torch.no_grad()
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor):
        last_scores = scores[-1]
        self.vars.push(last_scores.max(1)[0].float().cpu())
        self.varvars.push(self.vars.variance())
        self.size += 1
        if self.size < self.window_size:
            return False

        varvar = self.varvars.variance()
        for b in range(len(last_scores)):
            if varvar[b] < self.threshold:
                if self.stop_inds[b] > 0 and not self.stopped[b]:
                    self.stopped[b] = self.stop_inds[b] >= self.size
                else:
                    self.stop_inds[b] = int(
                        min(max(self.size, 1) * 1.15 + 150 + self.window_size, 4095)
                    )
            else:
                self.stop_inds[b] = 0
                self.stopped[b] = False
        return all(self.stopped.values()) and len(self.stopped) > 0

In [10]:
class PDFLoader:
    """
    Load PDF to a custom document element format for vector databases.

    Args:
        pdf_path (Path): The path to the PDF file.

    Attributes:
        elements: Document elements to add to vector databases.
        pipe: Initiate table-transformer-detection.
    """
    def __init__(self, pdf_path: Path):
        self.pdf_path = pdf_path
        self.elements = []
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        self.pipe = pipeline("object-detection", model="microsoft/table-transformer-detection", device=self.device)
        self.processor_noug = AutoProcessor.from_pretrained("facebook/nougat-small")
        self.model_noug = VisionEncoderDecoderModel.from_pretrained("facebook/nougat-small")

    def load(self):
        images, filepath = rasterize_paper(self.pdf_path, return_pil=True)
        if images is None:
            print("Failed to rasterize PDF.")
            return []

        self.extract(images, filepath)
        print("elements stored")
        self.create_overlapping_pages(1000)
        print("elements overlapped")
        self.summarize_tables()
        print("tables summarized")

        return self.elements

    def extract(self, images, filepath):
        """
        Extract tables and texts from all images.
        """
        for i, image in enumerate(images):
            pixel_values = self.processor_noug(images=image, return_tensors="pt").pixel_values
            outputs = self.model_noug.generate(pixel_values.to("cpu"),
                                min_length=1,
                                max_length=3584,
                                bad_words_ids=[[self.processor_noug.tokenizer.unk_token_id]],
                                return_dict_in_generate=True,
                                output_scores=True,
                                stopping_criteria=StoppingCriteriaList([StoppingCriteriaScores()]),)
            generated = self.processor_noug.batch_decode(outputs[0], skip_special_tokens=True)[0]
            generated = self.processor_noug.post_process_generation(generated, fix_markdown=False)
            metadata = {"filepath": filepath, "page_number": i}
            self.elements.append(Element(type="text", page_content=generated, metadata=metadata))

        for ele in self.elements:
            if ele.type == "text":
                text = ele.page_content
                pattern = r'(\\begin{tabular}(.*?)\\end{tabular})'
                matches = re.findall(pattern, text, re.DOTALL)
                if matches:
                    for match in matches:
                        txt = text.replace(match[0], "")
                        self.elements.append(Element(type="table", page_content=match[0].strip(), metadata=ele.metadata))
                        # print(txt)
                        ele.page_content = txt
    

    def summarize_tables(self):
        llm = ChatOllama(model="phi3:medium-128k", temperature=0)
        prompt_text = """
        You are an assistant tasked with summarizing tables. \n 
        Give a detailed summary of the table. Do not use your pre-conceived notion and summarize exclusively with the information in the table. It is very important that you only provide the final output without any additional comments or remarks. Table: {element} 
        """
        prompt = ChatPromptTemplate.from_template(prompt_text)
        summarize_chain = {"element": lambda x: x} | prompt | llm | StrOutputParser()

        for element in self.elements:
            if element.type == "table":
                summary = summarize_chain.invoke({"element": str(element.page_content)})
                element.page_content = summary

    def create_overlapping_pages(self, overlap_size=1000):
        num_elements = len(self.elements)

        for i in range(num_elements - 1):
            if self.elements[i].metadata["source"] == "text":
                current_page_content = self.elements[i].page_content
                next_page_content = self.elements[i + 1].page_content
                overlap_content = next_page_content[:overlap_size]
                combined_content = current_page_content + ' ' + overlap_content
                self.elements[i].page_content = combined_content

## Initialize embeddings
The following code is used to initialize the embeddings, `ACUNAO-Data` folder, text splitter, and Chroma vector database. 

In [11]:
# Import libraries
import os
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils import embedding_functions

In [12]:
def initialize_embeddings_and_db(folder_name):
    # Set the chunk size and overlap for text splitting
    chunk_size = 4500
    chunk_overlap = 1000

    # Specify the desktop path and folder name for vector database storage
    desktop_path = os.path.join(os.path.expanduser("~"), "Documents", "ACUNAO-Data")
    vdb_name = "vectordb"
    folder_path = os.path.join(desktop_path, folder_name, vdb_name)

    # Create the folder if it doesn't exist
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    
    # Initialize embeddings
    embeddings = embedding_functions.SentenceTransformerEmbeddingFunction("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

    # Initialize text splitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

    # Initialize Chroma vector store or load existing if available
    client = chromadb.PersistentClient(folder_path)
    collection = client.get_or_create_collection(name="acunao-db", embedding_function=embeddings)

    return embeddings, client, collection, text_splitter

## Document Processing
The following code observes the folder, `ACUNAO-Data`, and processes the documents added, modified, or deleted in the folder. 

In [13]:
# Import libraries
import os
from langchain_community.vectorstores.utils import filter_complex_metadata
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
import uuid
from pathlib import Path    
import logging
import time
import json
from datetime import datetime
from pytz import timezone
from docx2pdf import convert
import shutil

In [14]:
class DocumentEventHandler(FileSystemEventHandler):
    """
    A custom event handler for monitoring and processing document-related events in a file system. This handler processes files with specific extensions and triggers actions on create, modify, and delete events.
    
    Attributes:
        processor: An instance responsible for processing and updating the vector database.
        supported_extensions: A set of file extensions that the handler will process.
        process_start: A flag indicating the processing state.
    """
    def __init__(self, processor):
        self.processor = processor
        self.process_start = False
        self.process_end = False
        self.del_process_start = False
        self.del_process_end = False

    def on_any_event(self, event):
        normalized_path = os.path.normpath(event.src_path)
        path_parts = normalized_path.split(os.sep)
        
        if event.is_directory or not event.src_path.endswith(tuple(self.processor.supported_extensions)) or self.should_ignore(event.src_path):
            return None

        database_index = path_parts.index("ACUNAO-Data")
        subpath_parts = path_parts[database_index + 1:]
        if subpath_parts and os.path.splitext(subpath_parts[-1])[1]:
            subpath_parts = subpath_parts[:-1]
        self.processor.folder_name = os.sep.join(subpath_parts)

        if len(self.processor.folder_name) != 0: 
            if event.event_type in ['created', 'modified']:
                if not self.process_start:  # Only process if not already processing
                    self.processor.embeddings, self.processor.client, self.processor.vectordb, self.processor.text_splitter = initialize_embeddings_and_db(self.processor.folder_name)
                    self.processor.update_vector_db(event.src_path)

            elif event.event_type == 'deleted':
                self.del_process_start = True
                self.processor.embeddings, self.processor.client, self.processor.vectordb, self.processor.text_splitter = initialize_embeddings_and_db(self.processor.folder_name)
                self.processor.delete_from_vector_db(event.src_path)
                self.del_process_end = True

    def should_ignore(self, path):
        # Ignore files ending with '.tmp' or starting with '~'
        filename = os.path.basename(path)
        if filename.endswith('.tmp') or filename.startswith('~'):
            return True
        return False

In [15]:
class DocumentProcessor:
    """
    A class responsible for processing documents, managing embeddings, and interfacing with a vector database. This class initializes necessary components and sets up a file system observer for monitoring changes in the specified folder path.

    Attributes:
        desktop_path: Path to the ACUNAO-Data folder.
        folder_name: Name of selected folder within ACUNAO-Data.
        folder_path: Path to the folder containing documents to be processed.
        vectordb: Path to the vector database.
        embeddings: Placeholder for document embeddings.
        text_splitter: Placeholder for a text splitting utility.
        files: List to hold the names of the files to be processed.
        observer: Observer for monitoring file system changes.
        event_handler: Event handler for processing document-related events.
        observer_initialized: Flag indicating whether the observer has been initialized.
        observer_thread: Thread for running the observer.
        supported_extensions: List of file extensions that the processor will handle.
        timezone: Timezone to handle dates.
    """
    def __init__(self):
        self.desktop_path = os.path.join(os.path.expanduser("~"), "Documents", "ACUNAO-Data")
        self.folder_name = "project_example" 
        self.folder_path = os.path.join(self.desktop_path, self.folder_name)
        self.vectordb = None
        self.embeddings = None
        self.text_splitter = None
        self.files = []
        self.observer = None
        self.event_handler = None
        self.observer_initialized = False
        self.observer_thread = None
        self.supported_extensions = [".pdf", ".docx"]
        self.timezone = timezone('America/New_York') # Datetime defaults to EST

    def save_file_metadata(self, metadata, metadata_file):
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=4)

    def add_file_metadata(self, metadata, filename, database, datenow, timenow, modified_at, clock_time, cpu_time, size, filetype):
        metadata[filename] = {
            "database": database,
            "date_added": datenow,
            "time_added": timenow,
            "modified_at": modified_at,
            "clock_time": f"{clock_time}s",
            "cpu_time": f"{cpu_time}s",
            "size": f"{size}kb",
            "filetype": filetype
        }   

    def initialize_observer(self):
        if not self.observer_initialized:
            self.observer = Observer()
            self.event_handler = DocumentEventHandler(self)
            self.observer.schedule(self.event_handler, self.desktop_path, recursive=True)
            self.observer.start()
            self.observer_initialized = True

    def update_vector_db(self, file_path):
        file_extension = os.path.splitext(file_path)[1]

        if file_extension == ".docx":
            filename = Path(file_path).name
            database = os.path.dirname(os.path.abspath(file_path))
            metadata_file = os.path.join(database, "metadata.json")
            size = os.path.getsize(file_path)/1000
            modified_at = time.strftime('%Y-%m-%dT%H:%M:%S', time.localtime(os.path.getmtime(file_path)))

            # Load existing metadata if the file exists
            if os.path.exists(metadata_file):
                with open(metadata_file, 'r') as f:
                    metadata = json.load(f)
            else:
                metadata = {}

            # Skip files that have already been processed
            if filename in metadata and metadata[filename]["modified_at"] == modified_at:
                logging.info("Skipping already processed file: %s", file_path)
                return
            
            else: 
                start_time = time.time()
                t1_start = time.process_time() 

                convert(file_path)

                end_time = time.time()
                t1_stop = time.process_time()

                clock_time = "{:.2f}".format(end_time - start_time)
                cpu_time = "{:.2f}".format(t1_stop - t1_start)
                datenow, timenow = datetime.now(self.timezone).isoformat().split("T")

                self.add_file_metadata(metadata, filename, database, datenow, timenow, modified_at, clock_time, cpu_time, size, file_extension)
                self.save_file_metadata(metadata, metadata_file)
        
        elif file_extension == ".pdf":
            filename = Path(file_path).name
            database = os.path.dirname(os.path.abspath(file_path))
            metadata_file = os.path.join(database, "metadata.json")
            size = os.path.getsize(file_path)/1000
            modified_at = time.strftime('%Y-%m-%dT%H:%M:%S', time.localtime(os.path.getmtime(file_path)))
        
            # Load existing metadata if the file exists
            if os.path.exists(metadata_file):
                with open(metadata_file, 'r') as f:
                    metadata = json.load(f)
            else:
                metadata = {}

            # Skip files that have already been processed
            if filename in metadata and metadata[filename]["modified_at"] == modified_at:
                logging.info("Skipping already processed file: %s", file_path)
                return

            else:
                print("Changes detected in folder. Updating vector database...")
                self.event_handler.process_start = True
                start_time = time.time()
                t1_start = time.process_time() 

                logging.info("Processing file: %s", file_path)

                loader = PDFLoader(file_path)

                documents = loader.load()
                text_chunks = filter_complex_metadata(self.text_splitter.split_documents(documents))

                self.vectordb.add(
                    documents=[doc.page_content for doc in text_chunks],
                    metadatas=[doc.metadata for doc in text_chunks],
                    ids=[str(uuid.uuid4()) for _ in range(len(text_chunks))]
                )

                logging.info("File processed: %s", file_path)
                end_time = time.time()
                t1_stop = time.process_time()

                clock_time = "{:.2f}".format(end_time - start_time)
                cpu_time = "{:.2f}".format(t1_stop - t1_start)
                datenow, timenow = datetime.now(self.timezone).isoformat().split("T")

                self.add_file_metadata(metadata, filename, database, datenow, timenow, modified_at, clock_time, cpu_time, size, file_extension)
                self.save_file_metadata(metadata, metadata_file)
                self.event_handler.process_end = True

    def delete_from_vector_db(self, file_path):
        filename = Path(file_path).name
        database = os.path.dirname(os.path.abspath(file_path))
        metadata_file = os.path.join(database, "metadata.json")

        # Load existing metadata if the file exists
        if os.path.exists(metadata_file):
            with open(metadata_file, 'r') as f:
                metadata = json.load(f)
        else:
            metadata = {}

        if filename in metadata:
            print(f"File deleted: {file_path}. Removing from vector database...")
            self.vectordb.delete(where={"source": file_path})

            del metadata[filename]

            self.save_file_metadata(metadata, metadata_file)

    def run(self):
        if self.folder_name == "project_example":
            src_folder_path = "../data/2_test_data"
            dest_folder_path = os.path.join(self.desktop_path, self.folder_name)
            if not os.path.exists(dest_folder_path):
                os.makedirs(dest_folder_path)
            for item in os.listdir(src_folder_path):
                s = os.path.join(src_folder_path, item)
                d = os.path.join(dest_folder_path, item)
                # Skip .md files
                if os.path.isfile(s) and s.endswith('.md'):
                    continue
                # Copy the files to the directory
                if os.path.isdir(s):
                    shutil.copytree(s, d, dirs_exist_ok=True)
                else:
                    shutil.copy2(s, d)

        self.embeddings, self.client, self.vectordb, self.text_splitter = initialize_embeddings_and_db(self.folder_name)
        if not self.observer_initialized:
            self.initialize_observer()

        try:
            print("Running document processor. Press Ctrl+C to stop.")
            while True:
                time.sleep(1)
        except KeyboardInterrupt:
            print("Interrupted by user. Stopping...")
        except Exception as error:
            print("Error processing documents: " + str(error))
        finally:
            self.stop_observer()

    def stop_observer(self):
        if self.observer_initialized:
            self.observer.stop()
            self.observer.join()
            print("Stopped the observer and saved state.")
            self.observer_initialized = False

## Configure LLM
The following code configures the LLM. It contains system prompts, callbacks, and the retrieval chain. 

In [16]:
# Import libraries
import sys
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.chat_models import ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
import ollama

In [17]:
def init_llm():
    try:
        model_list = ollama.list()
        if "phi3:medium-128k" not in model_list:
            ollama.pull("phi3:medium-128k")
    except Exception as e:
        print(f"An error occurred: {e}")

In [18]:
class ChatPDFAssistant:
    """Handles PDF ingestion, query processing, and answering queries using a chat model."""

    def __init__(self, db="project_example", embeddings=None):
        # Initialize embeddings and vector database
        _, self.client, self.vectordb, self.text_splitter = initialize_embeddings_and_db(db)


        self.db = Chroma(client=self.client, collection_name="acunao-db",embedding_function=embeddings)


        # Initialize the language model
        self.llm = ChatOllama(model="phi3:medium-128k", temperature=0)


        self.DEFAULT_SYSTEM_PROMPT = """
        You are a good, honest project assistant. 

        If a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct. If you do not know the answer to a question, make it clear you do not know the answer instead of making up false information.
        """.strip()

        self.SYSTEM_PROMPT = "Use the following pieces of context to answer the question at the end. You must only answer within the provided context. If you do not know the answer, just say you don't know, don't try to make up an answer."

        self.template = self.generate_prompt(
            """
            {context}

            Question: {question}
            """,
            system_prompt=self.SYSTEM_PROMPT,
        )

        self.qa_prompt = PromptTemplate(template=self.template, input_variables=['context', 'question'])

        # Initialize the QA chain
        self.chain = RetrievalQA.from_chain_type(llm=self.llm,
                                                 chain_type='stuff',
                                                 retriever=self.db.as_retriever(search_type="similarity", search_kwargs={"k": 3}),
                                                 return_source_documents=True,
                                                 chain_type_kwargs={'prompt': self.qa_prompt},
                                                 verbose=True)

    def generate_prompt(self, prompt: str, system_prompt: str) -> str:
        return f"""
        [INST] <<SYS>>
        {system_prompt}
        <</SYS>>

        {prompt} [/INST]
        """.strip()

    def format_response(self, dictionary):
        """
        Formats the response dictionary to:
        - Print metadata for each document.
        - Display page_content text for PDF and Word documents.
        - Display the overall result after the document details.
        Assumes each document in source_documents is an instance of a Document class.
        """
        # Correctly define source_documents from the dictionary
        source_documents = dictionary["source_documents"]

        sources = "### Source Documents:\n"
        for doc in source_documents:
            # Safely get the 'source' and 'page' from metadata, default if not found
            file_directory = doc.metadata.get("filepath", "File directory not available.")
            # filename = doc.metadata.get("filename", "Filename not available.")
            source_path = file_directory
            page_number = doc.metadata.get("page_number", "Page number not available.")
            file_extension = file_directory.split('.')[-1].lower() if file_directory else ""
            
            # Metadata information
            metadata_info = f"**Source**: {source_path}\n**Page**: {page_number}\n"

            if file_extension in ["pdf", "doc", "docx"]:
                # Display page_content text
                page_content_text = doc.page_content.replace('\n', ' ') if doc.page_content else "Page content not available."
                sources += f"\n\n{metadata_info}\n{page_content_text}\n\n"
            else:
                # Fallback for other file types or if page_content should be displayed by default
                page_content_text = doc.page_content.replace('\n', ' ') if doc.page_content else "Page content not available."
                sources += f"\n\n{metadata_info}\n{page_content_text}\n\n"

        # Now appending the formatted result at the end
        formatted_result = dictionary["result"]
        complete_response = sources + "\n\n---\n\n### Result:\n" + formatted_result

        return complete_response

    def chat(self, input_text):
        user_input = str(input_text)
        if user_input == 'exit':
            print('Exiting')
            sys.exit()
        if user_input == '':
            return None
        result = self.chain.invoke({'query': user_input})
        formatted_response = self.format_response(result)
        return print(formatted_response)

## Setup all functionalities
The following code sets up and initializes the functionalities needed to interact with the llm.

In [19]:
# Import libraries
import threading

In [20]:
# Specify the desktop path and folder name for files storage
desktop_path = os.path.join(os.path.expanduser("~"), "Documents")
folder_name = "ACUNAO-Data"
folder_path = os.path.join(desktop_path, folder_name)

In [21]:
# Create the folder if it doesn't exist
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

In [22]:
processor = DocumentProcessor()

In [23]:
# Initiate document processor and thread
processor_thread = threading.Thread(target=processor.run, daemon=True)

WARNING: **DO NOT LEAVE THE PROCESSOR THREAD RUNNING FOREVER! PLEASE RESTART THE JUPYTER NOTEBOOK ONCE YOU ARE FINISHED WORKING ON THE NOTEBOOK.**

In [24]:
# This works and the observer and document processor will work in the background
# Go to the ACUNAO-Data folder in the Documents folder to see if the document is processed or not
# The document has finished processing if it is listed in the metadata.json file
processor_thread.start()

A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- configuration_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- modeling_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
<All keys matched successfully>


Running document processor. Press Ctrl+C to stop.


In [25]:
# Initiate the llm 
init_llm()

In [26]:
# If this is modified, make sure to change the embedding model used in other parts of the code as well
embeddings = SentenceTransformerEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5", model_kwargs={"trust_remote_code":True})

<All keys matched successfully>


## Chat with Your Document
The following code is used to interact with the LLM. It can be used to experiment and test the functionality of the llm. 

In [27]:
# The class takes two parameters: db and embeddings, set the db as the database you would like to query
# If the db is a sub-folder within a project folder, it should be written as "project_folder/subfolder"
assistant = ChatPDFAssistant("project_example", embeddings)

In [28]:
# Tip: If you would like to have the output easier to the eyes, word wrap the output. You can do this in VS Code by clicking the checkbox for Notebook > Output: Word Wrap

In [29]:
# Example query
assistant.chat("What is the student health insurance plan rate for the 2023-2024 academic for an international student?")



> Entering new RetrievalQA chain...

> Finished chain.
### Source Documents:


**Source**: File directory not available.
**Page**: Page number not available.

Tuition may be paid at the Student Account Services Office located in the Ashe Building, online using CaneLink, or by mail. Payment after 5:00 pm can be done by using the drop box located at the Cashier's Desk in the Ashe Building (to the right of the main stairs). Students may also set up a monthly payment plan by contacting Student Accounts.  Some graduate programs may offer fee waivers. Please contact your program coordinator tor additional information. Fee waiver applications submitted past the deadline will not be considered.  Veterans Assistance  The University of Miami’s Veterans’ Attairs (VA) Office assists veterans and dependents of veterans who are entitled to VA educational benefits under Chapters 30, 31, 33, 35, 1606, or 1607. UM also participants in the Yellow Ribbon program for qualified Chapter 33 recipients. Any